# Engine walkthrough

The simulation engine in `src/lab/`: synthetic populations with stored latent parameters, randomization, outcome generation with a configurable true effect, and the validated statistical toolbox. Because the truth is stored, every estimate in this notebook can be compared to the exact answer.

In [1]:
import numpy as np

from lab.assignment import assign_users
from lab.populations import continuous_users, draw_continuous
from lab.stats import cuped_t_test, welch_t_test

## A clean experiment, start to finish

100,000 users, true effect fixed at 0.5 by construction.

In [2]:
users = continuous_users(100_000, mean=10.0, between_sd=2.0, within_sd=5.0, seed=0)
treated = assign_users(users, treat_fraction=0.5, seed=1)
y = draw_continuous(users, treated, effect=0.5, within_sd=5.0, seed=2)
result = welch_t_test(y[treated], y[~treated])
print(f'estimate {result.estimate:.3f}, true effect 0.500')
ci = f'[{result.ci_low:.3f}, {result.ci_high:.3f}]'
print(f'95% CI {ci}, p = {result.p_value:.2e}')

estimate 0.457, true effect 0.500
95% CI [0.390, 0.524], p = 5.22e-41


## CUPED with the pre-period covariate

A persistent engagement metric (user-level signal comparable to period noise) with a 28-day pre-period aggregate as the covariate. The pre/post correlation is known analytically, so CUPED's variance reduction has an exact expected value: rho squared.

In [3]:
b, w, k = 3.0, 3.0, 28  # between-user sd, within sd, pre-period days
users2 = continuous_users(100_000, between_sd=b, within_sd=w,
                          pre_period_days=k, seed=3)
treated2 = assign_users(users2, seed=4)
y2 = draw_continuous(users2, treated2, effect=0.2, within_sd=w, seed=5)
raw = welch_t_test(y2[treated2], y2[~treated2])
cuped = cuped_t_test(y2[treated2], users2['pre_metric'][treated2],
                     y2[~treated2], users2['pre_metric'][~treated2])
rho = b**2 / (np.sqrt(b**2 + w**2 / k) * np.sqrt(b**2 + w**2))
print(f'measured variance reduction {cuped.variance_reduction:.3f}, '
      f'analytic rho^2 {rho**2:.3f}')
raw_width = raw.ci_high - raw.ci_low
cuped_width = cuped.test.ci_high - cuped.test.ci_low
print(f'CI width {raw_width:.4f} raw vs {cuped_width:.4f} CUPED '
      f'({1 - cuped_width / raw_width:.0%} narrower)')

measured variance reduction 0.483, analytic rho^2 0.483
CI width 0.1051 raw vs 0.0756 CUPED (28% narrower)
